In [ ]:
import os
import gc
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import *
from catboost import CatBoostClassifier, Pool

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 300)


class CFG:
    seed = 42

    target = "PitNextLap"
    id_col = "id"

    comp_paths = [
        "/kaggle/input/competitions/playground-series-s6e5",
        "/kaggle/input/playground-series-s6e5",
    ]

    original_paths = [
        "/kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv",
        "/kaggle/input/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv",
    ]

    use_original = True

    n_splits = 10

    task_type = "GPU"
    devices = "0:1"

    loss_function = "Logloss"
    eval_metric = "AUC"
    custom_metric = ["AUC", "Logloss"]

    iterations = 11000
    learning_rate = 0.018
    depth = 8

    l2_leaf_reg = 8.5
    random_strength = 0.65
    bagging_temperature = 0.45

    bootstrap_type = "Bayesian"
    grow_policy = "SymmetricTree"

    border_count = 254
    one_hot_max_size = 10

    min_data_in_leaf = 48
    max_ctr_complexity = 4
    ctr_leaf_count_limit = 64

    random_seed = 42
    early_stopping_rounds = 500
    use_best_model = True
    verbose = 500

    clip_low = 1e-7
    clip_high = 1 - 1e-7

    plot_top_n = 50


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


def first_existing_path(paths):
    for p in paths:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"No valid path found from: {paths}")


def reduce_mem_usage(df):
    out = df.copy()

    for col in out.columns:
        dtype = out[col].dtype

        if pd.api.types.is_integer_dtype(dtype):
            c_min, c_max = out[col].min(), out[col].max()

            if c_min >= np.iinfo(np.int8).min and c_max <= np.iinfo(np.int8).max:
                out[col] = out[col].astype(np.int8)
            elif c_min >= np.iinfo(np.int16).min and c_max <= np.iinfo(np.int16).max:
                out[col] = out[col].astype(np.int16)
            elif c_min >= np.iinfo(np.int32).min and c_max <= np.iinfo(np.int32).max:
                out[col] = out[col].astype(np.int32)

        elif pd.api.types.is_float_dtype(dtype):
            out[col] = pd.to_numeric(out[col], downcast="float")

    return out


def safe_divide(a, b, eps=1e-6):
    return a / (b + eps)


def print_section(title):
    print("\n" + "=" * 60)
    print(title)
    print("=" * 60)


seed_everything(CFG.seed)


print_section("Loading Data")

COMP_PATH = first_existing_path(CFG.comp_paths)

train_raw = pd.read_csv(f"{COMP_PATH}/train.csv")
test_raw = pd.read_csv(f"{COMP_PATH}/test.csv")
sample_submission = pd.read_csv(f"{COMP_PATH}/sample_submission.csv")

original_raw = None

if CFG.use_original:
    for p in CFG.original_paths:
        if os.path.exists(p):
            original_raw = pd.read_csv(p)
            print(f"Loaded original data: {original_raw.shape}")
            break

print(f"Train shape: {train_raw.shape}")
print(f"Test shape : {test_raw.shape}")
print(f"Target rate: {train_raw[CFG.target].mean():.6f}")

train_raw = reduce_mem_usage(train_raw)
test_raw = reduce_mem_usage(test_raw)

if original_raw is not None:
    original_raw = reduce_mem_usage(original_raw)


BASE_CAT_COLS = ["Driver", "Compound", "Race"]

NUMERIC_BASE_COLS = [
    "Year",
    "PitStop",
    "LapNumber",
    "Stint",
    "TyreLife",
    "Position",
    "LapTime (s)",
    "LapTime_Delta",
    "Cumulative_Degradation",
    "RaceProgress",
    "Position_Change",
]


def add_domain_features(df):
    out = df.copy()
    eps = 1e-6

    for col in BASE_CAT_COLS:
        if col in out.columns:
            out[col] = out[col].astype("string").fillna("__MISSING__").astype(str)

    if {"LapNumber", "RaceProgress"}.issubset(out.columns):
        out["EstimatedTotalLaps"] = safe_divide(
            out["LapNumber"],
            out["RaceProgress"].clip(lower=eps),
            eps,
        ).replace([np.inf, -np.inf], np.nan).clip(1, 120)

        out["LapsRemaining"] = (
            out["EstimatedTotalLaps"] - out["LapNumber"]
        ).clip(lower=0)

        out["LapProgress_x_LapNumber"] = out["LapNumber"] * out["RaceProgress"]

    if {"TyreLife", "LapNumber"}.issubset(out.columns):
        out["TyreAgeRatio"] = safe_divide(out["TyreLife"], out["LapNumber"].clip(lower=1), eps)
        out["LapPerTyreLife"] = safe_divide(out["LapNumber"], out["TyreLife"] + 1, eps)
        out["LapMinusTyreLife"] = out["LapNumber"] - out["TyreLife"]
        out["TyreLifeMinusLap"] = out["TyreLife"] - out["LapNumber"]

    if {"TyreLife", "EstimatedTotalLaps"}.issubset(out.columns):
        out["TyreAgeVsRace"] = safe_divide(out["TyreLife"], out["EstimatedTotalLaps"].clip(lower=1), eps)

    if {"TyreLife", "RaceProgress"}.issubset(out.columns):
        out["PitWindowPressure"] = out["TyreLife"] * out["RaceProgress"]
        out["TyreLife_x_RaceProgress"] = out["TyreLife"] * out["RaceProgress"]

    if {"TyreLife", "LapsRemaining"}.issubset(out.columns):
        out["TyreLife_to_LapsRemaining"] = safe_divide(out["TyreLife"], out["LapsRemaining"] + 1, eps)
        out["LapsRemaining_to_TyreLife"] = safe_divide(out["LapsRemaining"], out["TyreLife"] + 1, eps)

    if {"Cumulative_Degradation", "TyreLife"}.issubset(out.columns):
        out["DegPerTyreLap"] = safe_divide(out["Cumulative_Degradation"], out["TyreLife"].clip(lower=1), eps)
        out["AbsDegPerTyreLap"] = safe_divide(out["Cumulative_Degradation"].abs(), out["TyreLife"].clip(lower=1), eps)

    if {"Cumulative_Degradation", "LapNumber"}.issubset(out.columns):
        out["DegPerRaceLap"] = safe_divide(out["Cumulative_Degradation"], out["LapNumber"].clip(lower=1), eps)

    if {"LapTime_Delta", "TyreLife"}.issubset(out.columns):
        out["DeltaPerTyreLap"] = safe_divide(out["LapTime_Delta"], out["TyreLife"].clip(lower=1), eps)
        out["AbsDeltaPerTyreLap"] = safe_divide(out["LapTime_Delta"].abs(), out["TyreLife"].clip(lower=1), eps)

    if "LapTime_Delta" in out.columns:
        out["DeltaAbs"] = out["LapTime_Delta"].abs()
        out["LapTimeDeltaPositive"] = (out["LapTime_Delta"] > 0).astype(np.int8)
        out["LapTimeDeltaNegative"] = (out["LapTime_Delta"] < 0).astype(np.int8)

    if "Cumulative_Degradation" in out.columns:
        out["Abs_Cumulative_Degradation"] = out["Cumulative_Degradation"].abs()
        out["Positive_Degradation"] = (out["Cumulative_Degradation"] > 0).astype(np.int8)

    if "Position_Change" in out.columns:
        out["Abs_Position_Change"] = out["Position_Change"].abs()
        out["Gained_Position"] = (out["Position_Change"] > 0).astype(np.int8)
        out["Lost_Position"] = (out["Position_Change"] < 0).astype(np.int8)

    if {"Position", "RaceProgress"}.issubset(out.columns):
        out["PositionPressure"] = out["Position"] * out["RaceProgress"]

    if {"Stint", "TyreLife"}.issubset(out.columns):
        out["StintPressure"] = out["Stint"] * out["TyreLife"]
        out["TyreLife_x_Stint"] = out["TyreLife"] * out["Stint"]

    if {"Stint", "LapNumber"}.issubset(out.columns):
        out["Stint_x_LapNumber"] = out["Stint"] * out["LapNumber"]
        out["Is_First_Stint"] = (out["Stint"] == 1).astype(np.int8)
        out["Is_Late_Stint"] = (out["Stint"] >= 3).astype(np.int8)

    if "RaceProgress" in out.columns:
        out["Early_Race"] = (out["RaceProgress"] <= 0.25).astype(np.int8)
        out["Mid_Race"] = (
            (out["RaceProgress"] > 0.25) &
            (out["RaceProgress"] <= 0.65)
        ).astype(np.int8)
        out["Late_Race"] = (out["RaceProgress"] > 0.65).astype(np.int8)

        out["RacePhase"] = pd.cut(
            out["RaceProgress"],
            bins=[-np.inf, 0.20, 0.40, 0.60, 0.80, np.inf],
            labels=["P1", "P2", "P3", "P4", "P5"],
        ).astype(str)

    if "LapNumber" in out.columns:
        out["LapBin"] = pd.cut(
            out["LapNumber"],
            bins=[-np.inf, 5, 10, 20, 35, 50, np.inf],
            labels=["L_000_005", "L_006_010", "L_011_020", "L_021_035", "L_036_050", "L_051_plus"],
        ).astype(str)

    if "TyreLife" in out.columns:
        out["TyreLifeBin"] = pd.cut(
            out["TyreLife"],
            bins=[-np.inf, 3, 7, 12, 20, 30, np.inf],
            labels=["T_000_003", "T_004_007", "T_008_012", "T_013_020", "T_021_030", "T_031_plus"],
        ).astype(str)

    if "Position" in out.columns:
        out["PositionBin"] = pd.cut(
            out["Position"],
            bins=[-np.inf, 3, 8, 14, np.inf],
            labels=["front", "upper_mid", "lower_mid", "back"],
        ).astype(str)

    def make_cross(name, cols):
        if set(cols).issubset(out.columns):
            val = out[cols[0]].astype(str)
            for c in cols[1:]:
                val = val + "_" + out[c].astype(str)
            out[name] = val

    make_cross("Race_Year", ["Race", "Year"])
    make_cross("Compound_Stint", ["Compound", "Stint"])
    make_cross("Driver_Race", ["Driver", "Race"])
    make_cross("Driver_Compound", ["Driver", "Compound"])
    make_cross("Race_Compound", ["Race", "Compound"])
    make_cross("Race_Compound_Stint", ["Race", "Compound", "Stint"])
    make_cross("Compound_RacePhase", ["Compound", "RacePhase"])
    make_cross("Compound_TyreLifeBin", ["Compound", "TyreLifeBin"])
    make_cross("RacePhase_TyreLifeBin", ["RacePhase", "TyreLifeBin"])

    out = out.replace([np.inf, -np.inf], np.nan)

    for col in out.select_dtypes(include=["float64"]).columns:
        out[col] = out[col].astype(np.float32)

    return out


def add_digit_features(df, numeric_cols=None, int_digit_limit=3, decimal_digit_limit=2):
    out = df.copy()

    if numeric_cols is None:
        numeric_cols = out.select_dtypes(include=[np.number]).columns.tolist()

    for col in numeric_cols:
        if col not in out.columns:
            continue

        s = out[col].fillna(0).astype(float)
        abs_s = s.abs()

        for i in range(int_digit_limit):
            new_col = f"{col}_int_digit_{i + 1}"
            out[new_col] = ((abs_s // (10 ** i)) % 10).astype(np.int8)

        if pd.api.types.is_float_dtype(out[col]):
            for i in range(1, decimal_digit_limit + 1):
                new_col = f"{col}_dec_digit_{i}"
                out[new_col] = ((abs_s * (10 ** i)).round().astype(int) % 10).astype(np.int8)

    return out


def add_float_signature_features(df, float_cols=None):
    out = df.copy()

    if float_cols is None:
        float_cols = out.select_dtypes(include=["float32", "float64"]).columns.tolist()

    selected = [
        "RaceProgress",
        "LapTime (s)",
        "LapTime_Delta",
        "Cumulative_Degradation",
        "TyreAgeRatio",
        "DegPerTyreLap",
        "DegPerRaceLap",
        "DeltaPerTyreLap",
        "DeltaAbs",
        "PitWindowPressure",
        "EstimatedTotalLaps",
        "LapsRemaining",
        "LapMinusTyreLife",
    ]

    float_cols = [c for c in selected if c in out.columns]

    for col in float_cols:
        scaled = (out[col].fillna(0).astype(float) * 100).round().astype(int).abs()

        for i in range(5):
            new_col = f"{col}_sig_{i + 1}"
            digit = ((scaled // (10 ** i)) % 10).astype(np.int8)

            if digit.nunique() > 1:
                out[new_col] = digit.astype(str)

    return out


def add_string_precision_features(df):
    out = df.copy()

    if "RaceProgress" in out.columns:
        out["RaceProgress_str"] = out["RaceProgress"].round(4).astype(str)

    if "EstimatedTotalLaps" in out.columns:
        out["EstimatedTotalLaps_str"] = out["EstimatedTotalLaps"].round(1).astype(str)

    if "TyreAgeRatio" in out.columns:
        out["TyreAgeRatio_str"] = out["TyreAgeRatio"].round(3).astype(str)

    return out


def add_frequency_features(train_df, test_df, original_df=None):
    freq_cols = [
        "Driver",
        "Race",
        "Compound",
        "Race_Year",
        "Compound_Stint",
        "Driver_Race",
        "Driver_Compound",
        "Race_Compound",
        "Race_Compound_Stint",
        "Compound_RacePhase",
        "Compound_TyreLifeBin",
        "RacePhase_TyreLifeBin",
        "LapBin",
        "TyreLifeBin",
        "PositionBin",
    ]

    freq_cols = [c for c in freq_cols if c in train_df.columns and c in test_df.columns]

    frames = [train_df, test_df]
    if original_df is not None:
        frames.append(original_df)

    base = pd.concat([f[freq_cols] for f in frames], axis=0, ignore_index=True)
    total = len(base)

    for col in freq_cols:
        counts = base[col].astype(str).value_counts(dropna=False)

        for f in frames:
            f[f"{col}_count"] = f[col].astype(str).map(counts).fillna(0).astype(np.float32)
            f[f"{col}_freq"] = (f[f"{col}_count"] / total).astype(np.float32)

    del base
    gc.collect()

    if original_df is not None:
        return train_df, test_df, original_df

    return train_df, test_df, None


def add_light_group_stats(train_df, test_df, original_df=None):
    group_cols_list = [
        ["Race_Year"],
        ["Race_Compound_Stint"],
        ["Driver_Race"],
        ["Compound_Stint"],
    ]

    value_cols = [
        "LapTime_Delta",
        "Position_Change",
        "RaceProgress",
        "TyreLife",
    ]

    frames = [train_df, test_df]
    if original_df is not None:
        frames.append(original_df)

    for group_cols in group_cols_list:
        group_cols = [c for c in group_cols if all(c in f.columns for f in frames)]

        if not group_cols:
            continue

        group_name = "_".join(group_cols)

        for value_col in value_cols:
            if not all(value_col in f.columns for f in frames):
                continue

            base = pd.concat(
                [f[group_cols + [value_col]] for f in frames],
                axis=0,
                ignore_index=True,
            )

            stats = (
                base.groupby(group_cols, dropna=False)[value_col]
                .agg(["mean", "std"])
                .reset_index()
            )

            stats.columns = group_cols + [
                f"{value_col}_mean_by_{group_name}",
                f"{value_col}_std_by_{group_name}",
            ]

            for idx, f in enumerate(frames):
                frames[idx] = f.merge(stats, on=group_cols, how="left")

                mean_col = f"{value_col}_mean_by_{group_name}"
                frames[idx][f"{value_col}_diff_mean_by_{group_name}"] = (
                    frames[idx][value_col] - frames[idx][mean_col]
                ).astype(np.float32)

            del base, stats
            gc.collect()

    if original_df is not None:
        return frames[0], frames[1], frames[2]

    return frames[0], frames[1], None


def clean_columns(train_df, test_df, original_df=None):
    train_df = train_df.copy()
    test_df = test_df.copy()

    if original_df is not None:
        original_df = original_df.copy()

    common_cols = [c for c in train_df.columns if c in test_df.columns]

    train_df = train_df[common_cols + [CFG.target]]
    test_df = test_df[common_cols]

    if original_df is not None:
        for c in common_cols:
            if c not in original_df.columns:
                original_df[c] = np.nan

        original_df = original_df[common_cols + [CFG.target]]

    return train_df, test_df, original_df


def fill_missing_and_types(train_df, test_df, original_df=None):
    frames = [train_df, test_df]
    if original_df is not None:
        frames.append(original_df)

    all_feature_cols = [c for c in train_df.columns if c != CFG.target]

    cat_cols = []

    for col in all_feature_cols:
        if any(
            (
                f[col].dtype == "object"
                or str(f[col].dtype).startswith("category")
                or str(f[col].dtype).startswith("string")
            )
            for f in frames
            if col in f.columns
        ):
            cat_cols.append(col)

    num_cols = [c for c in all_feature_cols if c not in cat_cols]

    for col in cat_cols:
        values = pd.concat([f[col].astype("string") for f in frames if col in f.columns], axis=0)
        mode_value = values.mode().iloc[0] if len(values.mode()) else "__MISSING__"

        for f in frames:
            if col in f.columns:
                f[col] = f[col].astype("string").fillna(mode_value).astype(str)

    for col in num_cols:
        values = pd.concat([f[col] for f in frames if col in f.columns], axis=0)
        fill_value = values.median()

        for f in frames:
            if col in f.columns:
                f[col] = f[col].replace([np.inf, -np.inf], np.nan).fillna(fill_value)
                if f[col].dtype == "float64":
                    f[col] = f[col].astype(np.float32)

    train_df = reduce_mem_usage(train_df)
    test_df = reduce_mem_usage(test_df)

    if original_df is not None:
        original_df = reduce_mem_usage(original_df)

    return train_df, test_df, original_df, cat_cols


print_section("Feature Engineering")

train_fe = train_raw.copy()
test_fe = test_raw.copy()

if original_raw is not None and CFG.target in original_raw.columns:
    original_fe = original_raw.copy()
else:
    original_fe = None

if original_fe is not None and "Normalized_TyreLife" in original_fe.columns:
    original_fe = original_fe.drop(columns=["Normalized_TyreLife"])

train_fe["IsOriginalData"] = 0
test_fe["IsOriginalData"] = 0

if original_fe is not None:
    original_fe["IsOriginalData"] = 1

train_fe = add_domain_features(train_fe)
test_fe = add_domain_features(test_fe)

if original_fe is not None:
    original_fe = add_domain_features(original_fe)

digit_source_cols = [
    "Year",
    "PitStop",
    "LapNumber",
    "Stint",
    "TyreLife",
    "Position",
    "LapTime (s)",
    "LapTime_Delta",
    "Cumulative_Degradation",
    "RaceProgress",
    "Position_Change",
    "EstimatedTotalLaps",
    "LapsRemaining",
    "TyreAgeRatio",
    "DegPerTyreLap",
    "DegPerRaceLap",
    "DeltaPerTyreLap",
    "DeltaAbs",
    "PositionPressure",
    "StintPressure",
    "PitWindowPressure",
    "LapMinusTyreLife",
]

digit_source_cols = [c for c in digit_source_cols if c in train_fe.columns and c in test_fe.columns]

train_fe = add_digit_features(train_fe, digit_source_cols)
test_fe = add_digit_features(test_fe, digit_source_cols)

if original_fe is not None:
    original_fe = add_digit_features(original_fe, digit_source_cols)

train_fe = add_float_signature_features(train_fe)
test_fe = add_float_signature_features(test_fe)

if original_fe is not None:
    original_fe = add_float_signature_features(original_fe)

train_fe = add_string_precision_features(train_fe)
test_fe = add_string_precision_features(test_fe)

if original_fe is not None:
    original_fe = add_string_precision_features(original_fe)

train_fe, test_fe, original_fe = add_frequency_features(train_fe, test_fe, original_fe)
train_fe, test_fe, original_fe = add_light_group_stats(train_fe, test_fe, original_fe)

train_fe, test_fe, original_fe = clean_columns(train_fe, test_fe, original_fe)
train_fe, test_fe, original_fe, cat_cols = fill_missing_and_types(train_fe, test_fe, original_fe)

print(f"Train FE shape: {train_fe.shape}")
print(f"Test FE shape : {test_fe.shape}")

if original_fe is not None:
    print(f"Original FE shape: {original_fe.shape}")

print(f"Categorical features: {len(cat_cols)}")


print_section("Preparing 10-Fold CV Data")

X_comp = train_fe.drop(columns=[CFG.target, CFG.id_col], errors="ignore")
y_comp = train_fe[CFG.target].astype(int).reset_index(drop=True)

test_ids = test_raw[CFG.id_col].copy()
X_test_final = test_fe.drop(columns=[CFG.id_col], errors="ignore")

if original_fe is not None:
    X_orig = original_fe.drop(columns=[CFG.target, CFG.id_col], errors="ignore")
    y_orig = original_fe[CFG.target].astype(int).reset_index(drop=True)
else:
    X_orig = None
    y_orig = None

common_features = [c for c in X_comp.columns if c in X_test_final.columns]

X_comp = X_comp[common_features].reset_index(drop=True)
X_test_final = X_test_final[common_features].reset_index(drop=True)

if X_orig is not None:
    for c in common_features:
        if c not in X_orig.columns:
            X_orig[c] = np.nan

    X_orig = X_orig[common_features].reset_index(drop=True)

cat_cols = [c for c in cat_cols if c in common_features]
cat_features = [X_comp.columns.get_loc(c) for c in cat_cols]

print(f"Competition train shape: {X_comp.shape}")
print(f"Test shape             : {X_test_final.shape}")

if X_orig is not None:
    print(f"Original train shape   : {X_orig.shape}")

print(f"Competition target rate: {y_comp.mean():.6f}")
print(f"Number of features     : {X_comp.shape[1]}")
print(f"Categorical features   : {len(cat_features)}")

gc.collect()


def get_catboost_params(seed, iterations):
    params = {
        "iterations": iterations,
        "learning_rate": CFG.learning_rate,
        "depth": CFG.depth,
        "l2_leaf_reg": CFG.l2_leaf_reg,
        "random_strength": CFG.random_strength,
        "bootstrap_type": CFG.bootstrap_type,
        "bagging_temperature": CFG.bagging_temperature,
        "loss_function": CFG.loss_function,
        "eval_metric": CFG.eval_metric,
        "custom_metric": CFG.custom_metric,
        "auto_class_weights": "Balanced",
        "task_type": CFG.task_type,
        "devices": CFG.devices,
        "random_seed": seed,
        "early_stopping_rounds": CFG.early_stopping_rounds,
        "allow_writing_files": False,
        "verbose": CFG.verbose,
        "border_count": CFG.border_count,
        "one_hot_max_size": CFG.one_hot_max_size,
        "min_data_in_leaf": CFG.min_data_in_leaf,
        "max_ctr_complexity": CFG.max_ctr_complexity,
        "grow_policy": CFG.grow_policy,
    }

    if CFG.task_type == "CPU":
        params["thread_count"] = -1
        params["ctr_leaf_count_limit"] = CFG.ctr_leaf_count_limit

    return params


def find_best_threshold(y_true, preds):
    thresholds = np.linspace(0.05, 0.95, 181)

    best_threshold = 0.5
    best_f1 = -1

    for t in thresholds:
        pred_class = (preds >= t).astype(int)
        score = f1_score(y_true, pred_class)

        if score > best_f1:
            best_f1 = score
            best_threshold = t

    return best_threshold, best_f1


def print_prediction_distribution(name, preds):
    print_section(name)

    print(f"Min       : {preds.min():.6f}")
    print(f"1st perc : {np.percentile(preds, 1):.6f}")
    print(f"5th perc : {np.percentile(preds, 5):.6f}")
    print(f"25th perc: {np.percentile(preds, 25):.6f}")
    print(f"Median   : {np.median(preds):.6f}")
    print(f"75th perc: {np.percentile(preds, 75):.6f}")
    print(f"95th perc: {np.percentile(preds, 95):.6f}")
    print(f"99th perc: {np.percentile(preds, 99):.6f}")
    print(f"Max      : {preds.max():.6f}")
    print(f"Mean     : {preds.mean():.6f}")


def save_fold_metric_plot(fold_metrics_df, metric_col, filename, title):
    plt.figure(figsize=(10, 6))
    plt.plot(fold_metrics_df["fold"], fold_metrics_df[metric_col], marker="o", label=metric_col)
    plt.axhline(fold_metrics_df[metric_col].mean(), linestyle="--", label=f"Mean {metric_col}")
    plt.xlabel("Fold")
    plt.ylabel(metric_col)
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(filename, dpi=150)
    plt.close()


print_section("10-Fold CatBoost Training")

skf = StratifiedKFold(
    n_splits=CFG.n_splits,
    shuffle=True,
    random_state=CFG.seed,
)

oof_pred = np.zeros(len(X_comp), dtype=np.float32)
test_pred_folds = np.zeros((len(X_test_final), CFG.n_splits), dtype=np.float32)

fold_metrics = []
fold_importances = []

for fold, (trn_idx, val_idx) in enumerate(skf.split(X_comp, y_comp), 1):
    print_section(f"Fold {fold}/{CFG.n_splits}")

    X_tr_comp = X_comp.iloc[trn_idx].reset_index(drop=True)
    y_tr_comp = y_comp.iloc[trn_idx].reset_index(drop=True)

    X_val = X_comp.iloc[val_idx].reset_index(drop=True)
    y_val = y_comp.iloc[val_idx].reset_index(drop=True)

    if X_orig is not None:
        X_tr = pd.concat([X_tr_comp, X_orig], axis=0, ignore_index=True)
        y_tr = pd.concat([y_tr_comp, y_orig], axis=0, ignore_index=True)
    else:
        X_tr = X_tr_comp.copy()
        y_tr = y_tr_comp.copy()

    print(f"Train shape       : {X_tr.shape}")
    print(f"Validation shape  : {X_val.shape}")
    print(f"Train target rate : {y_tr.mean():.6f}")
    print(f"Valid target rate : {y_val.mean():.6f}")

    model = CatBoostClassifier(
        **get_catboost_params(CFG.seed + fold, CFG.iterations)
    )

    model.fit(
        X_tr,
        y_tr,
        eval_set=(X_val, y_val),
        cat_features=cat_features,
        use_best_model=True,
    )

    val_pred = model.predict_proba(X_val)[:, 1]
    val_pred = np.clip(val_pred, CFG.clip_low, CFG.clip_high)

    test_pred = model.predict_proba(X_test_final)[:, 1]
    test_pred = np.clip(test_pred, CFG.clip_low, CFG.clip_high)

    oof_pred[val_idx] = val_pred
    test_pred_folds[:, fold - 1] = test_pred

    best_threshold, best_f1 = find_best_threshold(y_val, val_pred)

    val_class_05 = (val_pred >= 0.5).astype(int)
    val_class_best = (val_pred >= best_threshold).astype(int)

    fold_auc = roc_auc_score(y_val, val_pred)
    fold_logloss = log_loss(y_val, val_pred)
    fold_ap = average_precision_score(y_val, val_pred)
    fold_brier = brier_score_loss(y_val, val_pred)

    fold_precision_05 = precision_score(y_val, val_class_05)
    fold_recall_05 = recall_score(y_val, val_class_05)
    fold_f1_05 = f1_score(y_val, val_class_05)

    fold_precision_best = precision_score(y_val, val_class_best)
    fold_recall_best = recall_score(y_val, val_class_best)
    fold_f1_best = f1_score(y_val, val_class_best)

    best_iter = model.get_best_iteration()
    if best_iter is None:
        best_iter = CFG.iterations

    print(f"Fold AUC             : {fold_auc:.6f}")
    print(f"Fold LogLoss         : {fold_logloss:.6f}")
    print(f"Fold AveragePrecision: {fold_ap:.6f}")
    print(f"Fold Brier Score     : {fold_brier:.6f}")
    print(f"Best iteration       : {best_iter}")
    print(f"Best F1 threshold    : {best_threshold:.4f}")
    print(f"F1 @ 0.5             : {fold_f1_05:.6f}")
    print(f"F1 @ best threshold  : {fold_f1_best:.6f}")

    fold_metrics.append({
        "fold": fold,
        "auc": fold_auc,
        "logloss": fold_logloss,
        "average_precision": fold_ap,
        "brier_score": fold_brier,
        "best_iteration": best_iter,
        "best_f1_threshold": best_threshold,
        "precision_at_05": fold_precision_05,
        "recall_at_05": fold_recall_05,
        "f1_at_05": fold_f1_05,
        "precision_at_best_threshold": fold_precision_best,
        "recall_at_best_threshold": fold_recall_best,
        "f1_at_best_threshold": fold_f1_best,
        "valid_size": len(X_val),
        "valid_target_rate": y_val.mean(),
        "test_pred_mean": test_pred.mean(),
    })

    fold_fi = pd.DataFrame({
        "feature": X_comp.columns,
        "importance": model.get_feature_importance(),
        "fold": fold,
    })

    fold_importances.append(fold_fi)

    del model, X_tr_comp, y_tr_comp, X_val, y_val, X_tr, y_tr
    gc.collect()


print_section("CV Results")

oof_pred = np.clip(oof_pred, CFG.clip_low, CFG.clip_high)
test_pred = np.clip(test_pred_folds.mean(axis=1), CFG.clip_low, CFG.clip_high)

fold_metrics_df = pd.DataFrame(fold_metrics)

print("\nFold Metrics:")
print(fold_metrics_df.to_string(index=False))

print("\nMetric Summary:")
summary_cols = [
    "auc",
    "logloss",
    "average_precision",
    "brier_score",
    "best_iteration",
    "best_f1_threshold",
    "f1_at_05",
    "f1_at_best_threshold",
    "test_pred_mean",
]

metric_summary = fold_metrics_df[summary_cols].agg(["mean", "std", "min", "max"]).T
print(metric_summary.to_string())


print_section("OOF Evaluation")

oof_auc = roc_auc_score(y_comp, oof_pred)
oof_logloss = log_loss(y_comp, oof_pred)
oof_ap = average_precision_score(y_comp, oof_pred)
oof_brier = brier_score_loss(y_comp, oof_pred)

best_oof_threshold, best_oof_f1 = find_best_threshold(y_comp, oof_pred)

oof_class_05 = (oof_pred >= 0.5).astype(int)
oof_class_best = (oof_pred >= best_oof_threshold).astype(int)

print(f"OOF AUC              : {oof_auc:.6f}")
print(f"OOF LogLoss          : {oof_logloss:.6f}")
print(f"OOF AveragePrecision : {oof_ap:.6f}")
print(f"OOF Brier Score      : {oof_brier:.6f}")

print("\nThreshold = 0.5")
print(f"Precision: {precision_score(y_comp, oof_class_05):.6f}")
print(f"Recall   : {recall_score(y_comp, oof_class_05):.6f}")
print(f"F1       : {f1_score(y_comp, oof_class_05):.6f}")

print("\nBest OOF F1 Threshold")
print(f"Threshold: {best_oof_threshold:.4f}")
print(f"Precision: {precision_score(y_comp, oof_class_best):.6f}")
print(f"Recall   : {recall_score(y_comp, oof_class_best):.6f}")
print(f"F1       : {f1_score(y_comp, oof_class_best):.6f}")

cm = confusion_matrix(y_comp, oof_class_best)

cm_df = pd.DataFrame(
    cm,
    index=["Actual 0", "Actual 1"],
    columns=["Predicted 0", "Predicted 1"],
)

print("\nOOF Confusion Matrix at Best F1 Threshold:")
print(cm_df.to_string())

print("\nOOF Classification Report:")
print(classification_report(y_comp, oof_class_best, target_names=["No Pit", "Pit"]))


print_prediction_distribution("OOF Prediction Distribution", oof_pred)
print_prediction_distribution("Test Prediction Distribution", test_pred)


print_section("Feature Importance")

all_fi = pd.concat(fold_importances, axis=0, ignore_index=True)

cv_fi = (
    all_fi
    .groupby("feature", as_index=False)
    .agg(
        importance_mean=("importance", "mean"),
        importance_std=("importance", "std"),
        importance_min=("importance", "min"),
        importance_max=("importance", "max"),
    )
    .sort_values("importance_mean", ascending=False)
)

print(cv_fi.head(50).to_string(index=False))


print_section("Saving Plots")

save_fold_metric_plot(
    fold_metrics_df,
    "auc",
    "cv_auc_by_fold.png",
    "CV AUC by Fold",
)

save_fold_metric_plot(
    fold_metrics_df,
    "logloss",
    "cv_logloss_by_fold.png",
    "CV LogLoss by Fold",
)

save_fold_metric_plot(
    fold_metrics_df,
    "f1_at_best_threshold",
    "cv_f1_by_fold.png",
    "CV F1 by Fold",
)

fpr, tpr, _ = roc_curve(y_comp, oof_pred)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f"OOF ROC AUC = {oof_auc:.6f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("OOF ROC Curve")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("oof_roc_curve.png", dpi=150)
plt.close()

precision_vals, recall_vals, _ = precision_recall_curve(y_comp, oof_pred)

plt.figure(figsize=(8, 6))
plt.plot(recall_vals, precision_vals, label=f"OOF AP = {oof_ap:.6f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("OOF Precision-Recall Curve")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("oof_precision_recall_curve.png", dpi=150)
plt.close()

plt.figure(figsize=(10, 6))
plt.hist(oof_pred[y_comp.values == 0], bins=60, alpha=0.6, label="Actual 0")
plt.hist(oof_pred[y_comp.values == 1], bins=60, alpha=0.6, label="Actual 1")
plt.xlabel("OOF Prediction")
plt.ylabel("Count")
plt.title("OOF Prediction Distribution by Actual Class")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("oof_prediction_distribution.png", dpi=150)
plt.close()

top_fi = cv_fi.head(CFG.plot_top_n).iloc[::-1]

plt.figure(figsize=(12, 14))
plt.barh(top_fi["feature"], top_fi["importance_mean"])
plt.xlabel("Mean Importance")
plt.ylabel("Feature")
plt.title(f"Top {CFG.plot_top_n} Mean Feature Importances")
plt.tight_layout()
plt.savefig("top_feature_importance.png", dpi=150)
plt.close()

print("Saved cv_auc_by_fold.png")
print("Saved cv_logloss_by_fold.png")
print("Saved cv_f1_by_fold.png")
print("Saved oof_roc_curve.png")
print("Saved oof_precision_recall_curve.png")
print("Saved oof_prediction_distribution.png")
print("Saved top_feature_importance.png")


print_section("Saving Outputs")

target_col = (
    CFG.target
    if CFG.target in sample_submission.columns
    else [c for c in sample_submission.columns if c != CFG.id_col][0]
)

submission = sample_submission.copy()
submission[target_col] = test_pred
submission.to_csv("submission.csv", index=False)

oof_predictions = pd.DataFrame({
    CFG.id_col: train_raw[CFG.id_col].values if CFG.id_col in train_raw.columns else np.arange(len(y_comp)),
    "y_true": y_comp.values,
    "oof_pred": oof_pred,
    "class_05": oof_class_05,
    "class_best_threshold": oof_class_best,
})

oof_predictions.to_csv("oof_predictions.csv", index=False)

test_fold_predictions = pd.DataFrame(
    test_pred_folds,
    columns=[f"fold_{i}" for i in range(1, CFG.n_splits + 1)]
)

test_fold_predictions.insert(0, CFG.id_col, test_ids.values)
test_fold_predictions["mean_prediction"] = test_pred
test_fold_predictions["std_prediction"] = test_pred_folds.std(axis=1)
test_fold_predictions.to_csv("test_fold_predictions.csv", index=False)

fold_metrics_df.to_csv("fold_metrics.csv", index=False)
all_fi.to_csv("fold_feature_importance.csv", index=False)
cv_fi.to_csv("cv_feature_importance.csv", index=False)

diagnostics = pd.DataFrame({
    "metric": [
        "n_splits",
        "oof_auc",
        "oof_logloss",
        "oof_average_precision",
        "oof_brier_score",
        "best_oof_f1_threshold",
        "f1_at_05",
        "f1_at_best_threshold",
        "precision_at_best_threshold",
        "recall_at_best_threshold",
        "mean_fold_auc",
        "std_fold_auc",
        "mean_fold_logloss",
        "std_fold_logloss",
        "mean_best_iteration",
        "std_best_iteration",
        "train_rows_competition",
        "train_rows_original",
        "test_rows",
        "n_features",
        "n_categorical_features",
        "test_pred_mean",
        "test_pred_std",
    ],
    "value": [
        CFG.n_splits,
        oof_auc,
        oof_logloss,
        oof_ap,
        oof_brier,
        best_oof_threshold,
        f1_score(y_comp, oof_class_05),
        f1_score(y_comp, oof_class_best),
        precision_score(y_comp, oof_class_best),
        recall_score(y_comp, oof_class_best),
        fold_metrics_df["auc"].mean(),
        fold_metrics_df["auc"].std(),
        fold_metrics_df["logloss"].mean(),
        fold_metrics_df["logloss"].std(),
        fold_metrics_df["best_iteration"].mean(),
        fold_metrics_df["best_iteration"].std(),
        len(X_comp),
        0 if X_orig is None else len(X_orig),
        len(X_test_final),
        X_comp.shape[1],
        len(cat_features),
        test_pred.mean(),
        test_pred.std(),
    ],
})

diagnostics.to_csv("diagnostics.csv", index=False)

print("Saved submission.csv")
print("Saved oof_predictions.csv")
print("Saved test_fold_predictions.csv")
print("Saved fold_metrics.csv")
print("Saved fold_feature_importance.csv")
print("Saved cv_feature_importance.csv")
print("Saved diagnostics.csv")

print_section("Finished")